In [ ]:
!pip install wikipedia
!pip install sentence_transformers

In [ ]:
import re
import pandas as pd
#from datasets import load_dataset
import time
import pickle
from tqdm import tqdm

In [ ]:
df=pd.read_pickle("/content/drive/MyDrive/NLP project/Abhishek (Mutli tool Chained- hotpot)/First_QAWC.pkl")

In [ ]:
df.head()

,Q,A,W,C
0,"The Dutch-Belgian television series that ""Hous...",6,House of Anubis > House of Anubis is a mystery...,First search for [Het Huis Anubis first aired ...
1,What would be the length of the track where th...,10.213,2013 Liqui Moly Bathurst 12 Hour > The 2013 Li...,First search [Mount Panorama Circuit -Wiki-> y...
2,"The 1988 American comedy film, The Great Outdo...",4,The Great Outdoors (film) > The Great Outdoors...,First Search [Annette Bening -Wiki-> y1]. She ...
3,"Dua Lipa, an English singer, songwriter and mo...",17,Dua Lipa > Her self-titled debut studio album...,First Search [ Dua Lipa New Rules release -Wik...
4,How old would be the female main protagonist o...,22,Catching Fire > Catching Fire is a 2009 scienc...,First Search for [Catching Fire Suzanne Collin...


In [ ]:
df.shape

(9225, 4)

In [ ]:
import requests
import json
from bs4 import BeautifulSoup
from pprint import pprint
import wikipedia

from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

from transformers import pipeline
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')


def wikipedia_seach(query):
  language_code = 'en'

  number_of_results = 3
  headers = {
    # 'Authorization': 'Bearer YOUR_ACCESS_TOKEN',
    'User-Agent': 'YOUR_APP_NAME (YOUR_EMAIL_OR_CONTACT_PAGE)'
  }

  base_url = 'https://api.wikimedia.org/core/v1/wikipedia/'
  endpoint = '/search/page'
  url = base_url + language_code + endpoint

  search_query = query
  parameters = {'q': search_query, 'limit': number_of_results}
  response = requests.get(url, headers=headers, params=parameters)
  response = json.loads(response.text)

  searched_titles=[]
  for page in range(len(response['pages'])):
    display_title = response['pages'][page]['title']
    searched_titles.append(display_title)
    response['pages'][page]["article_url"] = 'http://en.wikipedia.org/?curid=' + str(response['pages'][page]['id'])
    response['pages'][page]['excerpt'] = BeautifulSoup(response['pages'][page]['excerpt']).get_text()

  excerpt_text=search_query+": "+"\n".join([page['excerpt'] for page in response['pages']])
  if(len(searched_titles)!=0):
    #go to page whose title+ excerpt is most similar
    # Encode search query
    query_embedding = sentence_model.encode(search_query)
    page_texts = [page['title'] + ": " + page['excerpt'] for page in response['pages']]
    page_embeddings = sentence_model.encode(page_texts)
    # Calculate similarity scores for each page
    similarities = cosine_similarity([query_embedding], page_embeddings)[0]

    title_score_dict = dict(zip(searched_titles, similarities))

    # Get the index of the page with the highest similarity
    max_similarity_index = similarities.argmax()

    # Output the page id with the highest similarity
    max_similarity_page_id = response['pages'][max_similarity_index]['id']

    try:
      a=wikipedia.page(pageid=max_similarity_page_id)
      full_content=a.content
      return title_score_dict,excerpt_text,full_content
    except:
      return {},"",""
  else:
    return {},"",""

def QA(ques, context):
  result = question_answerer(question=ques,context=context)
  return result['answer'], result['score']

In [ ]:
## CLEANING

In [ ]:
row=0
print(df.loc[row]['Q'])
print(df.loc[row]['W'])
print(df.loc[row]['C'])
print(df.loc[row]['A'])

The Dutch-Belgian television series that "House of Anubis" was based on first aired in how many years after 2000 was?
House of Anubis > House of Anubis is a mystery television series developed for Nickelodeon based on the Dutch-Belgian television series "Het Huis Anubis". | Het Huis Anubis >  It first aired in September 2006 and the last episode was broadcast on December 4, 2009.
First search for [Het Huis Anubis first aired -Wiki-> y1]. First episode released in [y1 -QA(Which year was the fist episode aired ?)-> y2]. Number of years it aired after 2000 [y2 - 2000 = y3]. The answer is y3.
6


In [ ]:
pattern_rows = df[df['C'].str.contains(r'is \[y3\.\s*=\s*y4\] The answer is y4', regex=True)]
pattern_rows

,Q,A,W,C


In [ ]:
df['C'] = df['C'].str.replace(r'is \[y3\.\s*=\s*y4\] The answer is y4', 'is y3', regex=True)

### EXPERIMENT

In [ ]:
row=4
print(df.loc[row]['Q'])
print(df.loc[row]['W'])
print(df.loc[row]['C'])
print(df.loc[row]['A'])

How old would be the female main protagonist of Catching Fire, if she was born 6 years before?
Catching Fire > Catching Fire is a 2009 science fiction young adult novel by the American novelist Suzanne Collins, the second book in "The Hunger Games trilogy". | The Hunger Games (novel) >  It is written in the voice of 16-year-old Katniss Everdeen, who lives in the future, post-apocalyptic nation of Panem in North America.
First Search for [Catching Fire Suzanne Collins -Wiki-> y1]. Search for her age at that time [y1 -QA(What was the age of Katniss Everdeen at that time ?)-> y2]. She would how many years old is she was born 6 years before [y2 + 6 = y3]. The answer is y3.
22


In [ ]:
text=df.iloc[row]['C']
matches = re.findall(r'\[(.*?)\]', text)
matches

['Catching Fire Suzanne Collins -Wiki-> y1',
 'y1 -QA(What was the age of Katniss Everdeen at that time ?)-> y2',
 'y2 + 6 = y3']

In [ ]:
def process_list_tuple(rep): ## DECISION HERE
  if(str(type(rep[0]))!="<class 'tuple'>"): #if wikipedia article comes as in query, return empty string as it is anyways invalid
    return ""
  #for QA result, best score string returned
  non_empty_rep = [item for item in rep if item]  # Filter out empty tuples
  if non_empty_rep:
    rep = max(non_empty_rep, key=lambda x: x[1])[0]  # Get the string with the highest score
  else:
    rep = ""  # If all tuples are empty, return an empty string
  return str(rep)


def get_variables(text):
  matches = re.findall(r'\[(.*?)\]', text)
  variables={}
  wiki_searches=[]
  errors=[]

  def replace_variable(match):
    #print(variables)
    rep = variables.get(match.group(), match.group())
    if isinstance(rep, list):
        rep=process_list_tuple(rep)
    return rep

  for instruction in matches:
    print('at ', instruction)
    #---WIKI TOOL---
    if(instruction.find("-Wiki")!=-1):
      query,ans=instruction.split(" -Wiki-> ") #for in query variables
      in_query_var=re.search(r'y\d+', query)
      if(in_query_var):
        #print("IN QUERY VAR",query)
        query=re.sub(r'y\d+', replace_variable, query)
        #print(query)

      title_score_dict,excerpt_text,full_content=wikipedia_seach(query)#wikipedia search

      if title_score_dict:
        #print('HERE updating varibale of wiki answer',ans.strip())
        variables[ans.strip()] = [excerpt_text, full_content]#replaces the answer variable with extracted content
        wiki_searches.append(title_score_dict)
      else:
        errors.append(f"Error: Wikipedia search unsuccessful for query: '{query}'")
        continue

    #---QA TOOL---
    elif(instruction.find("-QA")!=-1):
      q,a=instruction.split("->")
      result = re.search(r'\((.*?)\)', q)
      if result:
          question = result.group(1)
      else:
          errors.append("Error in finding question")
          continue

      pre=q.split(" -QA")[0]
      context_exc=""
      full_context=""
      if (pre.find("+")!=-1): #if multiple contexts need to be appended

        vars=pre.split("+")
        for i in vars:
          if(i.strip() in variables.keys()):
            if((str(type(variables[i.strip()][0]))!="<class 'tuple'>")):
              context_exc=variables[i.strip()][0] #if variable came from wiki search, it would be a string else it would be a tuple
              full_context=variables[i.strip()][1]
            else:
              t=process_list_tuple(variables[i.strip()])
              #print('-----t',t)
              context_exc=t #if variable came from wiki search, it would be a string else it would be a tuple
              full_context=t
          else:
            errors.append(f"Error: Context variable '{i.strip()}' not found")
            continue

      else: #single context QA
        context_var=pre.strip()
        if context_var in variables.keys():
          if((str(type(variables[context_var][0]))!="<class 'tuple'>")):
            context_exc=variables[context_var][0] #if variable came from wiki search, it would be a string else it would be a tuple
            full_context=variables[context_var][1]
          else:
            t=process_list_tuple(variables[context_var])
            context_exc=t #if variable came from wiki search, it would be a string else it would be a tuple
            full_context=t
        else:
          errors.append(f"Error: Context variable '{context_var}' not found")
          continue

      execerpt_run=QA(question, context_exc)
      if(execerpt_run[-1]<0.75):
        full_run=QA(question, full_context)
      else:
        full_run=()

      #print('Updating variable',a.strip())
      variables[a.strip()]=[execerpt_run,full_run]

    #----MATH TOOL----
    else:
      print('going to match tool')
      all_eqs=[instruction]
      equation_variables = re.findall(r'y\d+', instruction)
      for var in equation_variables:
        if(var in variables.keys()):
          var_ans=process_list_tuple(variables[var])
          numbers = re.findall(r'\d+(?:\.\d+)?', var_ans)#we try searching for numbers
          if(len(numbers)>0):
            all_eqs.append(var+" = "+str(numbers[0])) #making equation for previously retrieved numerical values ### DECISiON HERE
            errors.append(f"WARNING: '{str(numbers)}'numbers in var ans")
          else:
            print('var ans is not a numeric',var_ans)
            errors.append(f"Error: '{var_ans}' not numeric")

      print(all_eqs)
      list_eq=[]
      for i in all_eqs:
        eq=sympify("Eq(" + i.replace("=", ",") + ")")
        list_eq.append(eq)

      result =solve(list_eq)
      print(result)
      for i in result.keys():
        if(str(i) not in variables.keys()):
          #print(i)
          variables[str(i)]=result[i]


  return variables,wiki_searches,errors


In [ ]:
v,w,e=get_variables(text)

at  Catching Fire Suzanne Collins -Wiki-> y1
at  y1 -QA(What was the age of Katniss Everdeen at that time ?)-> y2
at  y2 + 6 = y3
going to match tool
['y2 + 6 = y3', 'y2 = 1962']
{y2: 1962, y3: 1968}


In [ ]:
e

["WARNING: '['1962']'numbers in var ans"]

In [ ]:
v

In [ ]:
##sympy setup
!git clone https://github.com/sympy/sympy.git
import os
os.chdir('sympy/')
os.getcwdb()
!python -m pip install -e .